# A golden kernel on the FPGA

One notebook for every golden baseline (the kernel named like its model: `justounetsimple`,
`waveletcnn`, `sp_unet_small`, `justoliunet`, ...): plain loops, float, the model's int8
weights. The ARM does what the optimized kernels do in hardware: it drops the 10 unused
bands and z-scores the rest with `mu_sd.txt` (training's mean and std per band), and for a
2D network cuts the image into 32 x 32 patches (its last row and column repeated to fill
the edge patches). The FPGA computes the network: one patch, n patches or n pixels per
start, as the kernel's registers say. Its classes are checked against the image's labels.

Cloud / land / sea classification of a HYPSO capture: highest score = class:
**0 cloud, 1 land, 2 sea**.

Next to this notebook: the kernel's `.bit` + `.hwh`, its register map `x..._hw.h`,
`kernel.json` (which model it is; main.py writes it) and `mu_sd.txt`. And an image zip made
on the PC, the same one for every kernel:

    python3 tools/image.py prepare CAPTURE-l1a.nc --labels CAPTURE-l1a_labels.npy --crop 200 300 128 256 --out aegean_unet
    source fpga.sh; fpga_zip aegean_unet

The golden kernels are slow: crop the capture, or the whole image takes minutes. The
image zip can go in the folder above this one, once for every design. Run the cells from
the top.

## 1. Load the bitstream

In [ ]:
import json, re, time, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

# Which kernel: its register map, x<NAME>_hw.h, lies next to its NAME.bit; kernel.json says
# which model it is.
header_file = next(Path('.').glob('x*_hw.h'))
NAME = header_file.name[1:-len('_hw.h')]
info = json.loads(Path('kernel.json').read_text())
MODEL, P = info['model'], info['patch']         # P: the patch size, None for a 1D network
overlay = Overlay(f'{NAME}.bit')                 # also sets the FPGA clock from the .hwh
kernel = overlay.kernel                         # the HLS block, named 'kernel' in the block design

header = header_file.read_text()
def register(name):
    return int(re.search(rf'_ADDR_{name}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
AP_CTRL, DIN, DOUT = register('AP_CTRL'), register('DIN_DATA'), register('DOUT_DATA')
N = register('N_DATA') if '_ADDR_N_DATA' in header else None   # None: one patch per start

# The kernels' formats: in, float32 units preprocessed as in training: keep 110 of the 120
# raw bands, then (x - mean) / std; a unit is a 32 x 32 x 110 patch (2D) or a 110-band
# pixel (1D). Out, its float32 scores, 32 x 32 x 3 or 3.
BANDS, CLASSES = 110, 3
UNIT_IN, UNIT_OUT = ((P, P, BANDS), (P, P, CLASSES)) if P else ((BANDS,), (CLASSES,))
mu, sd = [np.array(a.replace(',', ' ').split(), dtype=np.float64)
          for a in re.findall(r'\[([^\]]*)\]', Path('mu_sd.txt').read_text())]
KEPT = [b for b in range(120) if b not in (0, 1, 2, 3, 4, 5, 6, 7, 118, 119)]
MEAN, INV_STD = mu.astype(np.float32), (1 / sd).astype(np.float32)

def launch(din_address, dout_address, n):
    # Start the kernel on n units; the addresses are 64 bits: low word, then high word.
    for offset, address in ((DIN, din_address), (DOUT, dout_address)):
        kernel.write(offset, address & 0xFFFFFFFF)
        kernel.write(offset + 4, address >> 32)
    if N is not None:
        kernel.write(N, n)
    kernel.write(AP_CTRL, 1)

def wait(timeout):
    deadline = time.monotonic() + timeout
    while not kernel.read(AP_CTRL) & 0x2:      # done
        if time.monotonic() > deadline:
            raise TimeoutError('kernel did not finish')

def run(units_in, units_out, n, timeout):
    # n units from units_in to units_out; the FPGA's time, start to done. A kernel without an
    # n register takes one patch per start: started on each in turn.
    t = time.perf_counter()
    if N is not None:
        launch(units_in.physical_address, units_out.physical_address, n)
        wait(timeout)
    else:
        size_in, size_out = units_in[0].nbytes, units_out[0].nbytes
        for k in range(n):
            launch(units_in.physical_address + k * size_in, units_out.physical_address + k * size_out, 1)
            wait(timeout)
    return time.perf_counter() - t

print(f'{NAME} ({MODEL}, {"2D, " + str(P) + " x " + str(P) + " patches" if P else "1D, pixels"}, '
      f'{"n" if N is not None else "one"} per start), idle:', bool(kernel.read(AP_CTRL) & 0x4))

## 2. Load the image and preprocess it into DDR

The preprocessed units go to DDR the FPGA can read (PYNQ's CMA, shared by every running
notebook): all of them at once if they fit, else a band of rows at a time.

In [ ]:
# The image: IMAGE.zip or the folder IMAGE/, next to this notebook or in the folder above
# it. Above is shared by every design's folder: upload the image there once. A zip is
# unpacked next to itself, and again only when it is newer than what was unpacked.
IMAGE = 'aegean_unet'
place = next((d for d in (Path('.'), Path('..'))
              if (d / f'{IMAGE}.zip').exists() or (d / IMAGE / 'cube.npy').exists()), None)
if place is None:
    raise FileNotFoundError(f'no {IMAGE}.zip or {IMAGE}/ in {Path(".").resolve()} or the folder above it')
folder, archive = place / IMAGE, place / f'{IMAGE}.zip'
if archive.exists() and (not (folder / 'cube.npy').exists()
                         or (folder / 'cube.npy').stat().st_mtime < archive.stat().st_mtime):
    with zipfile.ZipFile(archive) as z:
        z.extractall(place)
print(f'image: {folder.resolve()}')
t0 = time.monotonic()
cube = np.load(folder / 'cube.npy', mmap_mode='r')     # read from the SD card as it is used
meta = json.loads((folder / 'meta.json').read_text())
# The labels the classes are checked against: needed before the FPGA runs.
if not (folder / 'labels.npy').exists():
    raise FileNotFoundError(f'no labels.npy in {folder}: make the image with tools/image.py prepare --labels')
labels = np.load(folder / 'labels.npy')
if labels.shape != cube.shape[:2]:
    raise ValueError(f'labels.npy is {labels.shape}, the cube {cube.shape[:2]}')
H, W, _ = cube.shape

# A band: the image rows that make whole units. 2D: a row of patches (P rows; the last row
# and column repeat to fill the edge patches); 1D: any rows, their pixels.
BAND = P or 128
BANDS_ALL = -(-H // BAND)
UNITS_PER_BAND = -(-W // P) if P else BAND * W

def band_units(b, out):
    # Band b of the image, preprocessed, into out (units x UNIT_IN); returns how many units.
    y = b * BAND
    if P:
        rows = np.minimum(np.arange(y, y + P), H - 1)
        cols = np.minimum(np.arange(UNITS_PER_BAND * P), W - 1)
        strip = (cube[rows][:, cols][..., KEPT].astype(np.float32) - MEAN) * INV_STD
        out[:UNITS_PER_BAND] = strip.reshape(P, -1, P, BANDS).transpose(1, 0, 2, 3)
        return UNITS_PER_BAND
    pixels = np.asarray(cube[y:y + BAND]).reshape(-1, cube.shape[-1])
    np.multiply(pixels[:, KEPT].astype(np.float32) - MEAN, INV_STD, out=out[:len(pixels)])
    return len(pixels)

def band_scores(b, units_out, n):
    # The scores of band b's n units -> (its rows x W x 3).
    out = np.asarray(units_out[:n])
    if P:
        rows = out.transpose(1, 0, 2, 3).reshape(P, -1, CLASSES)
        return rows[:min(P, H - b * P), :W]
    return out.reshape(-1, W, CLASSES)

try:
    BANDS_PER_START = BANDS_ALL
    units_in = allocate(shape=(BANDS_ALL * UNITS_PER_BAND,) + UNIT_IN, dtype=np.float32)
    units_out = allocate(shape=(BANDS_ALL * UNITS_PER_BAND,) + UNIT_OUT, dtype=np.float32)
except Exception as error:                     # not enough (contiguous) CMA
    units_in = units_out = None
    BANDS_PER_START = 1
    units_in = allocate(shape=(UNITS_PER_BAND,) + UNIT_IN, dtype=np.float32)
    units_out = allocate(shape=(UNITS_PER_BAND,) + UNIT_OUT, dtype=np.float32)
    print(f'the whole image does not fit in CMA ({error}): a band of {BAND} rows per start')
COUNT, UNIT = (BANDS_ALL * UNITS_PER_BAND, 'patch') if P else (H * W, 'pixel')
print(f'{H} x {W} pixels' + (f', {COUNT} patches' if P else '') + f', loaded in {time.monotonic() - t0:.1f} s')

## 3. How long it will take: the first units, timed

In [ ]:
n = band_units(0, units_in)
test = 1 if P else min(n, 256)                 # one patch, or the first pixels
units_in.flush()                               # into DDR, not just the CPU cache
per_unit = run(units_in, units_out, test, timeout=600) / test   # also sets the whole image's timeouts
print(f'{1e3 * per_unit:.3f} ms per {UNIT} on the FPGA: the whole image will take about '
      f'{per_unit * COUNT:.1f} s')

## 4. The whole image

**Wall clock**: from calling `classify` until the scores are in place, the ARM's
preprocessing and patching included. **FPGA**: from the start bit until the done bit, as
the ARM sees it, added up over the starts.

In [ ]:
def classify():
    # The image -> (H x W x 3 float32 scores, seconds the FPGA ran).
    scores = np.zeros((H, W, CLASSES), dtype=np.float32)
    fpga = 0.0
    for first in range(0, BANDS_ALL, BANDS_PER_START):
        bands = range(first, min(first + BANDS_PER_START, BANDS_ALL))
        counts = [band_units(b, units_in[k * UNITS_PER_BAND:]) for k, b in enumerate(bands)]
        n = (len(bands) - 1) * UNITS_PER_BAND + counts[-1]
        units_in.flush()                       # into DDR, not just the CPU cache
        fpga += run(units_in, units_out, n, timeout=60 + 3 * per_unit * n)
        units_out.invalidate()                 # read what the kernel wrote
        for k, b in enumerate(bands):
            rows = band_scores(b, units_out[k * UNITS_PER_BAND:], counts[k])
            scores[b * BAND:b * BAND + len(rows)] = rows
        if BANDS_PER_START == 1:
            print(f'  band {first + 1} of {BANDS_ALL} done', flush=True)
    return scores, fpga

t = time.perf_counter()
scores, fpga = classify()
wall = time.perf_counter() - t
classes = scores.argmax(-1)
print(f'wall clock: {wall:9.3f} s for {H} x {W} pixels')
print(f'FPGA:       {fpga:9.3f} s, {1e3 * fpga / COUNT:.3f} ms per {UNIT}')
np.save(f'{IMAGE}_{NAME}_fpga_scores.npy', scores)    # next to this notebook: the image folder is shared

## 5. Check it against the labels

In [ ]:
known = labels < 3                             # 3 and above: unlabeled
print(f'accuracy against the labels: {(classes[known] == labels[known]).mean():.2%} of {known.sum()} labelled pixels')
for c, name in enumerate(('cloud', 'land', 'sea')):
    union = ((classes == c) | (labels == c))[known].sum()
    iou = ((classes == c) & (labels == c))[known].sum() / union if union else float('nan')
    print(f'  IoU {name}: {iou:.3f}')

In [ ]:
COLORS = np.array([[235, 235, 235], [60, 140, 60], [30, 90, 180], [128, 128, 128]], dtype=np.uint8)
rgb = np.asarray(cube[:, :, meta['rgb_bands']], dtype=np.float64)
lo, hi = np.percentile(rgb, [2, 98])
panels = {'picture': np.clip((rgb - lo) / (hi - lo), 0, 1),
          f'{NAME} classes': COLORS[classes]}
panels['labels'] = COLORS[np.minimum(labels, 3)]
wrong = np.full(labels.shape + (3,), 255, dtype=np.uint8)   # white: same class as the label
wrong[~known] = 128                                         # gray: unlabeled
wrong[known & (classes != labels)] = (220, 30, 30)          # red: a different class
panels['differs from labels'] = wrong
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4))
for ax, (title, img) in zip(axes, panels.items()):
    ax.imshow(img, cmap='gray_r' if img.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
plt.show()     # classes: cloud white, land green, sea blue, gray = unlabeled